# AMS 325 Final Project Group 17
Fight the Landlord: The Power of Probability

In [ ]:
class Card:
      def __init__(self, suit, rank, value):
              self.suit = suit   # The suit of the card (♠, ♥, ♦, ♣, or an empty string '' for Jokers)
              self.rank = rank   # The symbol displayed on the screen (3, 4, ..., J, Q, K, A, 2, Black_Joker, Red_Joker)
              self.value = value # The actual integer value used for comparing card sizes (3 is the smallest, Jokers are the largest)

      def __repr__(self):
        """
        Formats the card as a readable string when using the print() function. Example: [♠A], [♥3], [Red_Joker]
        """
        if self.suit == '':
            return f"[{self.rank}]"
        return f"[{self.suit}{self.rank}]"
      def __eq__(self, other):
        """
        Determines if two cards are exactly the same (allows the use of the == operator).
        This is essential for the Player class to precisely find and remove() specific cards from the hand.
        """
        if isinstance(other, Card):
              return self.rank == other.rank and self.suit == other.suit
        return False

      def __lt__(self, other):
        """
        Compares the size of the cards (allows the use of the < operator).
        By defining this, Python's built-in list sort() function can automatically
        arrange the cards properly based on their 'value' attribute.
        """
        if isinstance(other, Card):
              return self.value < other.value
        return NotImplemented

In [ ]:
class Player:
      def __init__(self, name):
        self.name = name
        self.hand = []        # The player's current cards (Stored in a Python built-in list)
        self.role = None      # Assigned as "Landlord" or "Farmer" during the bidding phase
        self.is_ai = False    # Flag to distinguish between a human player and an AI bot

      def sort_hand(self):
        """
        Sorts the cards in the hand in descending order (highest value first).
        This makes it easier for both the player and the AI to evaluate their hand.
        """
        self.hand.sort(key=lambda card: card.value, reverse=True)

      def remove_cards(self, cards_to_remove):
        """
        Removes the specified cards from the player's hand when they are played on the table.
        It uses object attribute comparison (rank and suit) to ensure the correct cards are removed.
        """
        for card in cards_to_remove:
        # Iterate through the hand to find the matching card object
          for h_card in self.hand:
            if h_card.rank == card.rank and h_card.suit == card.suit:
              self.hand.remove(h_card)
              break

      def get_hand_values(self):
        """
        Returns a list of the integer values of the cards currently in hand.
        This utility method is useful for quickly performing statistical or probabilistic calculations.
        """
        return [card.value for card in self.hand]

In [ ]:
from collections import Counter

class RuleEngine:
    # Definition of combination types
        SINGLE = "Single"
        PAIR = "Pair"
        TRIPLE = "Triple"
        BOMB = "Bomb"
        ROCKET = "Rocket" # A pair consisting of both the Black and Red Jokers
        INVALID = "Invalid"

        @staticmethod
        def identify_combination(cards):
          """
          Analyzes a list of card objects and returns the identified combination type
          along with its representative rank value.
          """
          n = len(cards)
          values = [card.value for card in cards]
          counts = Counter(values)
          distinct_values = sorted(counts.keys())

          # 1. Rocket (The strongest combination: both Black and Red Jokers)
          if n == 2 and 16 in values and 17 in values:
              return {"type": RuleEngine.ROCKET, "value": 17}

          # 2. Solo/Single (1 card)
          if n == 1:
              return {"type": RuleEngine.SINGLE, "value": values[0]}

          # 3. Pair (2 cards of the same rank)
          if n == 2 and len(counts) == 1:
              return {"type": RuleEngine.PAIR, "value": distinct_values[0]}

          # 4. Triple (3 cards of the same rank)
          if n == 3 and len(counts) == 1:
              return {"type": RuleEngine.TRIPLE, "value": distinct_values[0]}

          # 5. Bomb (4 cards of the same rank)
          if n == 4 and len(counts) == 1:
              return {"type": RuleEngine.BOMB, "value": distinct_values[0]}

          return {"type": RuleEngine.INVALID, "value": 0}

        @staticmethod
        def compare_plays(current_play_info, new_play_info):
          """
          Compares the current strongest play on the table with a new candidate play.
          Returns True if the new play is valid and stronger than the current one, False otherwise.
          """

           # If the table is empty, any valid combination can be played.
          if current_play_info is None:
             return new_play_info["type"] != RuleEngine.INVALID

          # A Rocket beats every other combination in the game.
          if new_play_info["type"] == RuleEngine.ROCKET:
              return True

          # If the current play on the table is a Rocket, no other play can beat it.
          if current_play_info["type"] == RuleEngine.ROCKET:
              return False

          # A Bomb beats any standard combination that is not a Bomb or a Rocket.
          if new_play_info["type"] == RuleEngine.BOMB and current_play_info["type"] != RuleEngine.BOMB:
              return True

           # For combinations of the same type, the new play must have a higher rank value.
           # Note: The number of cards in the combination must also be identical for a valid comparison.
          if new_play_info["type"] == current_play_info["type"]:
              return new_play_info["value"] > current_play_info["value"]

          return False

In [ ]:
import random

class Game:
    def __init__(self, players):
        self.players = players
        self.deck = []
        self.landlord_cards = []       # The 3 extra cards reserved for the Landlord
        self.current_turn = 0          # Index of the player whose turn it is
        self.table_cards = []          # Cards currently played on the table

    def initialize_deck(self):
        """
        Generates a standard 54-card deck (including Jokers) and shuffles it.
        """
        suits = ['♠', '♥', '♦', '♣']
        ranks_values = {
            '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9,
            '10': 10, 'J': 11, 'Q': 12, 'K': 13, 'A': 14, '2': 15
        }
        for suit in suits:
            for rank, value in ranks_values.items():
                self.deck.append(Card(suit, rank, value))
        self.deck.append(Card('', 'Black_Joker', 16))
        self.deck.append(Card('', 'Red_Joker', 17))
        random.shuffle(self.deck)

    def deal_cards(self):
        """
        Deals 17 cards to each of the 3 players and reserves 3 cards for the Landlord.
        All hands are sorted immediately after dealing for better organization.
        """
        for i in range(51):
            self.players[i % 3].hand.append(self.deck[i])

        self.landlord_cards = self.deck[51:]
        for player in self.players:
            player.sort_hand()

    def start_bidding(self):
        """
        Executes the bidding phase to determine the Landlord.
        Players bid 1, 2, or 3 points; the highest bidder becomes the Landlord.
        """
        print("\n--- Bidding Phase Starts ---")
        highest_bid = 0
        landlord_index = -1

        for i, player in enumerate(self.players):
            bid = random.choice([0, 1, 2, 3])
            print(f"{player.name} bids: {bid}")

            if bid > highest_bid:
                highest_bid = bid
                landlord_index = i
            if bid == 3:                  # Instant Landlord confirmation if someone bids 3
                break

        if landlord_index == -1:
            print("Everyone passed. Game needs to be redealt.")
            return False                 # Game restart required if no one bids

        # Assign the Landlord role and provide the extra cards
        landlord = self.players[landlord_index]
        landlord.role = "Landlord"
        landlord.hand.extend(self.landlord_cards)
        landlord.sort_hand()

        # Assign the Farmer role to the remaining players
        for i, player in enumerate(self.players):
            if i != landlord_index:
                player.role = "Farmer"

        self.current_turn = landlord_index      # The Landlord always starts the first turn
        print(f"\n{landlord.name} becomes the Landlord and gets the 3 extra cards!")
        print(f"Landlord's cards: {self.landlord_cards}")
        return True

    def game_loop(self):
        """게임의 메인 턴 진행 루프"""
        print("\n--- Game Starts ---")
        game_over = False
        consecutive_passes = 0           # Tracks the number of consecutive passes to reset the round
        current_winning_play = None      # Stores the current strongest combination on the table

        while not game_over:
            player = self.players[self.current_turn]
            print(f"\n[Turn] {player.name} ({player.role}, {len(player.hand)} cards left)")

            # If both opponents pass, the table is cleared and the last player to play starts a new round.
            if consecutive_passes == 2:
                print("Everyone passed. Table cleared. A new round starts.")
                current_winning_play = None
                consecutive_passes = 0

            move = None
            if current_winning_play is None:
                # Forced to play if starting a new round
                move = [player.hand.pop()]
            else:
                # If there is no card to hand in or stragically pass
                move = "Pass"

            if move == "Pass":
                print(f"{player.name} passes.")
                consecutive_passes += 1
            else:
                print(f"{player.name} plays: {move}")
                current_winning_play = move
                consecutive_passes = 0

            # Victory Condition: The first player to empty their hand wins the game.
            if len(player.hand) == 0:
                print(f"\n🎉 *** Game Over! {player.name} ({player.role}) wins! *** 🎉")
                game_over = True
                break

            # Cycle to the next player (0 -> 1 -> 2 -> 0)
            self.current_turn = (self.current_turn + 1) % 3

In [ ]:
# Create game instance and prepare for execution
p1 = Player("Hyeonwoo")
p2 = Player("Colin")
p3 = Player("AI_Bot")

# Create Game Object and Setup
dou_dizhu = Game([p1, p2, p3])
dou_dizhu.initialize_deck()
dou_dizhu.deal_cards()

# Start the auction and run the main game loop
if dou_dizhu.start_bidding():
    dou_dizhu.game_loop()



--- Bidding Phase Starts ---
Hyeonwoo bids: 0
Colin bids: 0
AI_Bot bids: 3

AI_Bot becomes the Landlord and gets the 3 extra cards!
Landlord's cards: [[♦K], [♠3], [♦Q]]

--- Game Starts ---

[Turn] AI_Bot (Landlord, 20 cards left)
AI_Bot plays: [[♠3]]

[Turn] Hyeonwoo (Farmer, 17 cards left)
Hyeonwoo passes.

[Turn] Colin (Farmer, 17 cards left)
Colin passes.

[Turn] AI_Bot (Landlord, 19 cards left)
Everyone passed. Table cleared. A new round starts.
AI_Bot plays: [[♦3]]

[Turn] Hyeonwoo (Farmer, 17 cards left)
Hyeonwoo passes.

[Turn] Colin (Farmer, 17 cards left)
Colin passes.

[Turn] AI_Bot (Landlord, 18 cards left)
Everyone passed. Table cleared. A new round starts.
AI_Bot plays: [[♠4]]

[Turn] Hyeonwoo (Farmer, 17 cards left)
Hyeonwoo passes.

[Turn] Colin (Farmer, 17 cards left)
Colin passes.

[Turn] AI_Bot (Landlord, 17 cards left)
Everyone passed. Table cleared. A new round starts.
AI_Bot plays: [[♦5]]

[Turn] Hyeonwoo (Farmer, 17 cards left)
Hyeonwoo passes.

[Turn] Colin (Fa